# Multi-Head Attention — Pure Python
# 
# This file builds on 01_attention_pure_python.md.
# Copy the utility functions and attention() from that file first,
# or run that notebook's cells before running this one.
#
# Here we implement:
#   1. A linear projection (learned weight matrix application)
#   2. Multi-Head Attention — splitting d_model into h heads
#   3. Feed-Forward Network — the MLP inside each Transformer block
#   4. Layer Normalization — normalizing each token independently
#   5. A full Encoder Block combining all of the above

import math
import random

# ─────────────────────────────────────────────────────────────
# Paste utilities from notebook 01 (or import them)
# ─────────────────────────────────────────────────────────────

def mat_mul(A, B):
    m, k, n = len(A), len(A[0]), len(B[0])
    result = [[0.0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            result[i][j] = sum(A[i][c] * B[c][j] for c in range(k))
    return result

def transpose(M):
    return [[M[r][c] for r in range(len(M))] for c in range(len(M[0]))]

def softmax(scores):
    m = max(scores)
    exps = [math.exp(s - m) for s in scores]
    t = sum(exps)
    return [e / t for e in exps]

def make_zeros(rows, cols):
    return [[0.0] * cols for _ in range(rows)]

def make_random_matrix(rows, cols, scale=0.1):
    return [[random.gauss(0, scale) for _ in range(cols)] for _ in range(rows)]

def add_vectors(a, b):
    return [x + y for x, y in zip(a, b)]

def attention(Q, K, V, mask=None):
    seq_q, d_k, d_v = len(Q), len(Q[0]), len(V[0])
    K_T = transpose(K)
    scores = mat_mul(Q, K_T)
    scale  = math.sqrt(d_k)
    scores = [[s / scale for s in row] for row in scores]
    if mask is not None:
        for i in range(len(scores)):
            for j in range(len(scores[0])):
                if mask[i][j] == 0:
                    scores[i][j] = float('-inf')
    weights = [softmax(row) for row in scores]
    output  = make_zeros(seq_q, d_v)
    for i in range(seq_q):
        for j in range(len(K)):
            for d in range(d_v):
                output[i][d] += weights[i][j] * V[j][d]
    return output, weights

# ─────────────────────────────────────────────────────────────
# SECTION 1: Linear Projection (Learned Weight Multiplication)
#
# In PyTorch you write:  output = nn.Linear(d_model, d_model)(input)
# What this actually does:
#   output[i] = W @ input[i] + b
# where W is a (d_out × d_in) weight matrix and b is a bias vector.
#
# For each token in the sequence, we apply the SAME weight matrix.
# This is what "position-wise" means in "position-wise linear projection."
# ─────────────────────────────────────────────────────────────

class Linear:
    """
    A learnable linear projection: output = input @ W.T + b
    
    W shape: (out_features, in_features)
    b shape: (out_features,)
    
    We transpose W when applying it so that each row of the input
    (one token's embedding) gets multiplied by the weight matrix.
    """
    def __init__(self, in_features, out_features):
        # Xavier initialization: scale by sqrt(2 / (in + out))
        # prevents activations from vanishing or exploding early in training
        scale = math.sqrt(2.0 / (in_features + out_features))
        self.W = make_random_matrix(out_features, in_features, scale=scale)
        self.b = [0.0] * out_features  # Biases start at zero

    def __call__(self, x):
        """
        x: a list of tokens, each token is a list of floats.
           Shape: [seq_len, in_features]
        Returns: [seq_len, out_features]
        
        For each token embedding (a row of x), we compute:
            output_token = W @ token + b
        """
        seq_len    = len(x)
        out_feat   = len(self.W)     # Number of output features
        in_feat    = len(self.W[0])  # Number of input features

        output = []
        for token in x:                   # Loop over each token in the sequence
            projected = [0.0] * out_feat
            for i in range(out_feat):     # For each output dimension
                s = self.b[i]             # Start with the bias
                for j in range(in_feat):  # Dot product with the weight row
                    s += self.W[i][j] * token[j]
                projected[i] = s
            output.append(projected)

        return output  # [seq_len, out_features]


# Test the Linear layer
random.seed(42)
linear = Linear(in_features=4, out_features=6)
x_test = make_random_matrix(3, 4, scale=1.0)  # 3 tokens, 4-dim each

out = linear(x_test)
print("Linear projection:")
print(f"  Input:  {len(x_test)} tokens × {len(x_test[0])} dims")
print(f"  Output: {len(out)} tokens × {len(out[0])} dims")

# ─────────────────────────────────────────────────────────────
# SECTION 2: Multi-Head Attention
#
# The core idea: instead of one attention computation in full d_model
# dimensions, we split d_model into h smaller "heads" and run h
# independent attention operations in parallel.
#
# Why? Each head can learn to focus on a different type of relationship:
#   Head 0 might learn: "what is the subject of this verb?"
#   Head 1 might learn: "what words modify this noun?"
#   Head 2 might learn: "which pronouns refer to which nouns?"
#
# All heads' outputs are concatenated and projected back to d_model.
# The total computation stays the same since d_k = d_model / n_heads.
#
# With loops, the "run in parallel" part just means a for-loop over heads.
# ─────────────────────────────────────────────────────────────

def split_heads(x, n_heads):
    """
    Split the last dimension of x (d_model) into (n_heads, d_k).
    
    x:       [seq_len, d_model]
    Returns: [n_heads, seq_len, d_k]
    
    We're rearranging the data so that each head gets a slice of
    the embedding dimension. Head 0 gets columns 0..d_k-1,
    Head 1 gets columns d_k..2*d_k-1, etc.
    
    In PyTorch this is done with .view() + .transpose() — here it's
    just list slicing.
    """
    seq_len = len(x)
    d_model = len(x[0])
    d_k     = d_model // n_heads

    heads = []
    for h in range(n_heads):
        start = h * d_k
        end   = start + d_k
        # Extract columns [start:end] for all tokens
        head_data = [token[start:end] for token in x]  # [seq_len, d_k]
        heads.append(head_data)

    return heads  # [n_heads, seq_len, d_k]


def concat_heads(heads):
    """
    Inverse of split_heads: concatenate the head outputs back together.
    
    heads:   [n_heads, seq_len, d_k]
    Returns: [seq_len, d_model]   where d_model = n_heads * d_k
    """
    n_heads = len(heads)
    seq_len = len(heads[0])

    result = []
    for t in range(seq_len):
        # For token t, concatenate the output of each head
        token_combined = []
        for h in range(n_heads):
            token_combined.extend(heads[h][t])  # Append this head's d_k values
        result.append(token_combined)

    return result  # [seq_len, n_heads * d_k = d_model]


class MultiHeadAttention:
    """
    Multi-Head Self-Attention — pure Python implementation.

    d_model: Full embedding dimension (e.g. 8)
    n_heads: Number of parallel attention heads (e.g. 2)
    d_k:     Per-head dimension = d_model // n_heads (e.g. 4)
    """
    def __init__(self, d_model, n_heads):
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads

        # Four projection matrices:
        # W_q, W_k, W_v: project from d_model → d_model
        # W_o: project the concatenated heads back to d_model
        self.W_q = Linear(d_model, d_model)
        self.W_k = Linear(d_model, d_model)
        self.W_v = Linear(d_model, d_model)
        self.W_o = Linear(d_model, d_model)

    def __call__(self, x, mask=None):
        """
        x:    [seq_len, d_model] — the input token embeddings (same for Q, K, V in self-attention)
        mask: [seq_len, seq_len] — 1=attend, 0=block

        Returns: [seq_len, d_model] — the attended-over, re-projected representation
        """
        seq_len = len(x)

        # ── Step 1: Project x into Q, K, V spaces ────────────
        # Each is still [seq_len, d_model], but in a LEARNED subspace
        Q = self.W_q(x)   # [seq_len, d_model]
        K = self.W_k(x)   # [seq_len, d_model]
        V = self.W_v(x)   # [seq_len, d_model]

        print(f"  After W_q projection: {len(Q)} tokens × {len(Q[0])} dims")

        # ── Step 2: Split each of Q, K, V into n_heads ───────
        Q_heads = split_heads(Q, self.n_heads)  # [n_heads, seq_len, d_k]
        K_heads = split_heads(K, self.n_heads)
        V_heads = split_heads(V, self.n_heads)

        print(f"  After split_heads: {self.n_heads} heads × {seq_len} tokens × {self.d_k} dims")

        # ── Step 3: Run attention independently per head ──────
        # This is the "parallel" part — in a real GPU these run simultaneously,
        # but in Python we just loop over heads sequentially.
        head_outputs = []
        all_weights  = []

        for h in range(self.n_heads):
            head_out, head_weights = attention(Q_heads[h], K_heads[h], V_heads[h], mask)
            # head_out shape: [seq_len, d_k]
            head_outputs.append(head_out)
            all_weights.append(head_weights)
            print(f"  Head {h} attention computed: {len(head_out)} tokens × {len(head_out[0])} dims")

        # ── Step 4: Concatenate all heads back together ───────
        combined = concat_heads(head_outputs)  # [seq_len, d_model]
        print(f"  After concat_heads: {len(combined)} tokens × {len(combined[0])} dims")

        # ── Step 5: Final output projection ───────────────────
        # This linear layer mixes information ACROSS heads, letting them
        # communicate and produce the final representation.
        output = self.W_o(combined)  # [seq_len, d_model]
        print(f"  After W_o projection: {len(output)} tokens × {len(output[0])} dims")

        return output, all_weights


# Test Multi-Head Attention
random.seed(7)
d_model = 8
n_heads = 2
seq_len = 4

mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
x   = make_random_matrix(seq_len, d_model, scale=1.0)

print("\n" + "=" * 55)
print(f"Multi-Head Attention: d_model={d_model}, n_heads={n_heads}, d_k={d_model//n_heads}")
print("=" * 55)
print(f"Input: {len(x)} tokens × {len(x[0])} dims\n")

output, weights = mha(x)

print(f"\nFinal output shape: {len(output)} × {len(output[0])}")
print(f"Attention weights per head: {len(weights)} heads, each {len(weights[0])}×{len(weights[0][0])}")

# ─────────────────────────────────────────────────────────────
# SECTION 3: Layer Normalization
#
# BatchNorm normalizes across the BATCH dimension (all samples).
# LayerNorm normalizes across the FEATURE dimension (per token).
#
# For each token embedding vector, we compute:
#   mean = average of all d_model values in this token
#   std  = standard deviation of those values
#   normalized[d] = (x[d] - mean) / (std + epsilon)
#
# Then two learned parameters gamma and beta let the network
# re-scale and re-shift after normalization:
#   output[d] = gamma[d] * normalized[d] + beta[d]
#
# This is critical: without gamma/beta, every token's embedding
# would be forced to have mean=0 and std=1, which might not be
# the best distribution for the task.
# ─────────────────────────────────────────────────────────────

class LayerNorm:
    """
    Layer Normalization — normalizes each token's embedding independently.
    
    In PyTorch: nn.LayerNorm(d_model)
    """
    def __init__(self, d_model, eps=1e-6):
        self.eps   = eps
        self.gamma = [1.0] * d_model  # Learned scale (starts at 1)
        self.beta  = [0.0] * d_model  # Learned shift (starts at 0)

    def __call__(self, x):
        """
        x: [seq_len, d_model]
        
        For each token (row), independently normalize its values.
        """
        output = []
        for token in x:
            n    = len(token)

            # Compute mean across d_model dimensions for this ONE token
            mean = sum(token) / n

            # Compute variance: average of squared deviations from mean
            variance = sum((v - mean) ** 2 for v in token) / n

            # Normalize: shift by mean, scale by standard deviation
            # eps prevents division by zero if all values happen to be identical
            std = math.sqrt(variance + self.eps)
            normalized = [(v - mean) / std for v in token]

            # Apply learned gamma (scale) and beta (shift) per dimension
            scaled = [self.gamma[d] * normalized[d] + self.beta[d]
                      for d in range(n)]
            output.append(scaled)

        return output


# Verify LayerNorm works
ln = LayerNorm(d_model=4)
x_test = [[10.0, 200.0, 5.0, 50.0], [1.0, 2.0, 3.0, 4.0]]

print("\n\n" + "=" * 55)
print("LayerNorm Demo")
print("=" * 55)
for i, token in enumerate(x_test):
    print(f"\nToken {i} before: {[round(v, 2) for v in token]}")
    mean = sum(token) / len(token)
    print(f"  mean = {mean:.2f}")

normed = ln(x_test)
for i, token in enumerate(normed):
    print(f"Token {i} after:  {[round(v, 4) for v in token]}")
    print(f"  mean ≈ {sum(token)/len(token):.6f}  (close to 0)")
    std = math.sqrt(sum((v - sum(token)/len(token))**2 for v in token) / len(token))
    print(f"  std  ≈ {std:.6f}   (close to 1)")

# ─────────────────────────────────────────────────────────────
# SECTION 4: Feed-Forward Network
#
# Each Transformer block contains a 2-layer MLP applied independently
# to each token position. The same weights are used for every position
# (like a 1D convolution with kernel_size=1).
#
# Structure:
#   token → Linear(d_model → d_ff) → GELU → Linear(d_ff → d_model)
#
# d_ff is typically 4 × d_model. This large "expansion" gives the
# network space to store "factual knowledge" — research shows that
# facts like "Paris is the capital of France" live in the FFN weights.
# ─────────────────────────────────────────────────────────────

def gelu(x):
    """
    GELU activation: Gaussian Error Linear Unit.
    Smoother version of ReLU, used in BERT, GPT, and modern Transformers.
    
    Approximation formula (exact computation uses the error function):
      GELU(x) ≈ 0.5 * x * (1 + tanh(√(2/π) * (x + 0.044715 * x³)))
    
    In PyTorch: nn.GELU()
    """
    return 0.5 * x * (1.0 + math.tanh(math.sqrt(2.0 / math.pi) * (x + 0.044715 * x**3)))


class FeedForward:
    """
    Position-wise Feed-Forward Network.
    Applied to each token INDEPENDENTLY — the weights are shared across positions.
    
    d_model: input/output dimension
    d_ff:    inner dimension (typically 4 × d_model)
    """
    def __init__(self, d_model, d_ff=None):
        d_ff = d_ff or 4 * d_model
        self.linear1 = Linear(d_model, d_ff)    # Expand: d_model → d_ff
        self.linear2 = Linear(d_ff, d_model)    # Contract: d_ff → d_model

    def __call__(self, x):
        """
        x: [seq_len, d_model]
        
        Each token goes through: Linear → GELU → Linear
        """
        # Step 1: Expand to d_ff
        hidden = self.linear1(x)                              # [seq_len, d_ff]
        # Step 2: Non-linearity (GELU)
        hidden = [[gelu(v) for v in token] for token in hidden]
        # Step 3: Contract back to d_model
        output = self.linear2(hidden)                         # [seq_len, d_model]
        return output


# Test FeedForward
ff = FeedForward(d_model=8, d_ff=32)
x_test = make_random_matrix(4, 8, scale=1.0)
ff_out = ff(x_test)
print(f"\n\nFeedForward: {len(x_test)}×{len(x_test[0])} → {len(ff_out)}×{len(ff_out[0])}")

# ─────────────────────────────────────────────────────────────
# SECTION 5: Full Encoder Block
#
# One Encoder Block = Two Sub-layers with Residual Connections:
#
#   x → LayerNorm → MultiHeadAttention → + x → LayerNorm → FFN → + x
#
# The "+ x" is the RESIDUAL CONNECTION (skip connection from ResNet).
# It means the block only needs to learn the CHANGE (residual) from
# the identity, not the full transformation. This makes very deep
# networks much easier to train — gradients flow through the "+" path.
#
# We use Pre-LayerNorm (norm before the sub-layer), which trains
# more stably than Post-LayerNorm (norm after).
# ─────────────────────────────────────────────────────────────

class EncoderBlock:
    """
    One Transformer Encoder Block (Pre-LayerNorm version).
    
    Processes a sequence and returns a sequence of the same shape.
    """
    def __init__(self, d_model, n_heads):
        self.attn  = MultiHeadAttention(d_model, n_heads)
        self.ff    = FeedForward(d_model)
        self.norm1 = LayerNorm(d_model)
        self.norm2 = LayerNorm(d_model)

    def __call__(self, x, mask=None):
        """
        x: [seq_len, d_model]
        
        Sub-layer 1: Normalize → Self-Attention → Residual Add
        Sub-layer 2: Normalize → Feed-Forward → Residual Add
        """
        seq_len  = len(x)
        d_model  = len(x[0])

        # ── Sub-layer 1: Self-Attention ───────────────────────
        normed1  = self.norm1(x)            # Normalize BEFORE attention (Pre-LN)
        attn_out, _ = self.attn(normed1, mask)

        # Residual: add the attention output to the ORIGINAL x, not the normed x
        # This is the skip connection — gradient flows directly through this addition
        x = [add_vectors(x[t], attn_out[t]) for t in range(seq_len)]

        # ── Sub-layer 2: Feed-Forward ─────────────────────────
        normed2  = self.norm2(x)            # Normalize BEFORE FFN (Pre-LN)
        ff_out   = self.ff(normed2)

        # Residual again
        x = [add_vectors(x[t], ff_out[t]) for t in range(seq_len)]

        return x


# Test the full Encoder Block
random.seed(0)
d_model = 8
n_heads = 2
seq_len = 3

block = EncoderBlock(d_model=d_model, n_heads=n_heads)
x     = make_random_matrix(seq_len, d_model, scale=1.0)

print("\n\n" + "=" * 55)
print(f"Full Encoder Block: d_model={d_model}, n_heads={n_heads}")
print("=" * 55)
print(f"Input shape: {len(x)} × {len(x[0])}\n")

out = block(x)

print(f"\nOutput shape: {len(out)} × {len(out[0])}")
print("Input → Output (first token):")
print(f"  IN:  {[round(v, 3) for v in x[0]]}")
print(f"  OUT: {[round(v, 3) for v in out[0]]}")
print("\nNote: Shape is PRESERVED — this is key to stacking blocks!")